# 🕵️‍♂️ Proyecto 3: Data Engineering & ETL - Detección de Fraude Financiero (AML)

Bienvenido a mi tercer proyecto del portafolio. En el mundo real, los datos nunca están listos para analizarse.

**El Caso de Negocio:** Simulando el fallo de servidores de pago en una pasarela financiera internacional, se generó un archivo de transacciones altamente corrupto (`transacciones_corruptas.csv`). 

**Mi Misión Analítica:**
- Extraer 2,500 intentos de transacciones con severos daños estructurales.
- Limpiar y unificar fechas mezcladas (formatos americanos, europeos y Unix Epoch).
- Normalizar divisas internacionales empleando Expresiones Regulares (Regex) en Pandas y convirtiéndolas a Float USD.
- Detectar y manejar Direcciones IP nulas e inválidas.
- Exportar la data limpia a una base de datos relacional local (`SQLite`).
- Ejecutar **consultas SQL avanzadas** para detectar patrones y horarios inusuales (Fraude).

A continuación presento todo el Pipeline ETL:

In [4]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_csv('../data/raw/transacciones_corruptas.csv')

print(f"Filas totales: {df_raw.shape[0]}")
print(f"Columnas: {df_raw.shape[1]}")

df_raw.sample(10)


Filas totales: 2500
Columnas: 6


,TransactionID,Timestamp_mixed,Merchant,Amount_Currency,Country_Raw,Customer_IP
905,TRX-00906,1689849900,MercadoLibre,"4.303,85 €",UK,907.681.0.1
183,TRX-00184,2023-12-05 08:05:00,Starbucks,"$ 1,871.46",CO,NaN
1852,TRX-01853,ERROR_FECHA,Airbnb,"802,18 €",JP,NaN
1566,TRX-01567,30-09-2023 05:01,Shell Gas,"$ 2,656.50",USA,240.129.81.123
231,TRX-00232,ERROR_FECHA,Alibaba,"2.931,10 €",GBR,119.62.74.179
496,TRX-00497,04/23/2023 13:38,Uber,"$ 3,893.00",Japón,148.204.9.141
0,TRX-00001,12/13/2023 12:31,Apple Store,"2.965,19 €",GBR,64.203.38.164
275,TRX-00276,2023-01-08 10:48:00,MercadoLibre,"973,58 €",CO,66.223.25.42
836,TRX-00837,12/01/2023 11:02,Shell Gas,¥ 186745,UK,NaN
2428,TRX-02429,NaN,Starbucks,¥ 142738,JP,197.65.115.11


## 🧹 Fase 1: Extracción y Evaluación Estructural

Comienzo el análisis utilizando el método `.info()` para evaluar qué tipos de datos detectó Python inicialmente y diagnosticar la gravedad de los valores NULOS (`NaN` o faltantes).

In [5]:
df_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   TransactionID    2395 non-null   str  
 1   Timestamp_mixed  2445 non-null   str  
 2   Merchant         2500 non-null   str  
 3   Amount_Currency  2441 non-null   str  
 4   Country_Raw      2362 non-null   str  
 5   Customer_IP      2238 non-null   str  
dtypes: str(6)
memory usage: 117.3 KB


## 🗓️ Fase 2: Transformación y Estandarización de Fechas (Timestamps)

El problema más grave del dataset: 4 formatos de fecha diferentes conviviendo en la misma columna. 

He desarrollado una función en Python combinando `re.match` y `pd.to_datetime` capaz de detectar dinámicamente si la fecha viene en formato de Segundos de Época UNIX (1687694200), formato americano (MM/DD/YYYY) u otros, normalizando todas a estándar internacional **ISO-8601**.

In [6]:
def limpiar_timestamp(valor):
    if pd.isna(valor) or valor == 'ERROR_FECHA':
        return pd.NaT
    
    valor_str = str(valor).strip()
    
    if re.match(r'^\d{9,11}$', valor_str):
        return pd.to_datetime(int(valor_str), unit='s')
    
    if re.match(r'^\d{2}/\d{2}/\d{4}', valor_str):
        return pd.to_datetime(valor_str, format='%m/%d/%Y %H:%M', errors='coerce')
    
    if re.match(r'^\d{2}-\d{2}-\d{4}', valor_str):
        return pd.to_datetime(valor_str, format='%d-%m-%Y %H:%M', errors='coerce')
    
    return pd.to_datetime(valor_str, errors='coerce')

df_raw['Timestamp_clean'] = df_raw['Timestamp_mixed'].apply(limpiar_timestamp)

df_raw['Hour'] = df_raw['Timestamp_clean'].dt.hour

total = len(df_raw)
rescatadas = df_raw['Timestamp_clean'].notna().sum()
perdidas = total - rescatadas
print(f"\u2705 Fechas rescatadas y normalizadas: {rescatadas:,} ({rescatadas/total*100:.1f}%)")
print(f"\u274c Fechas irrecuperables (marcaremos como NaT): {perdidas}")
print("\n--- Muestra de fechas limpias ---")
df_raw[['Timestamp_mixed', 'Timestamp_clean', 'Hour']].sample(8)


✅ Fechas rescatadas y normalizadas: 2,365 (94.6%)
❌ Fechas irrecuperables (marcaremos como NaT): 135

--- Muestra de fechas limpias ---


,Timestamp_mixed,Timestamp_clean,Hour
2009,06-06-2023 13:49,2023-06-06 13:49:00,13.0
2273,2023-11-14 03:05:00,2023-11-14 03:05:00,3.0
1913,2023-10-31 06:47:00,2023-10-31 06:47:00,6.0
2195,2023-11-02 00:48:00,2023-11-02 00:48:00,0.0
51,2023-06-08 02:32:00,2023-06-08 02:32:00,2.0
1751,1694835780,2023-09-16 03:43:00,3.0
564,2023-04-16 07:12:00,2023-04-16 07:12:00,7.0
2175,08/02/2023 04:48,2023-08-02 04:48:00,4.0


## 💰 Fase 3: Limpieza Matemática de Divisas Internacionales (Regex)

La matriz financiera cruda contiene dólares con símbolo inicial (`$ 3,020.23`), euros con comas en lugar de puntos (`1,232.19 €`), e Yenes enteros (`¥ 115702`).

**El enfoque técnico diseñado:** Mediante lambdas y Expresiones Regulares, el algoritmo limpia los caracteres sucios, unifica las comas decimales e invoca tasas de conversión aproximadas pre-configuradas para homogeneizar toda la columna a **USD float64** puro.

In [7]:
TASA_EUR_A_USD = 1.08
TASA_JPY_A_USD = 0.0067
TASA_GBP_A_USD = 1.27

def limpiar_monto_a_usd(valor):
    if pd.isna(valor):
        return np.nan
    
    valor_str = str(valor).strip()
    
    es_eur = '€' in valor_str
    es_jpy = '¥' in valor_str
    
    limpio = re.sub(r'[^\d,.]', '', valor_str).strip()
    
    if not limpio:
        return np.nan
    
    if re.match(r'^[\d.]+,\d{2}$', limpio):
        limpio = limpio.replace('.', '').replace(',', '.')
    else:
        limpio = limpio.replace(',', '')
    
    try:
        monto = float(limpio)
    except ValueError:
        return np.nan
    
    if es_eur:
        return round(monto * TASA_EUR_A_USD, 2)
    elif es_jpy:
        return round(monto * TASA_JPY_A_USD, 2)
    else:
        return round(monto, 2)

df_raw['Amount_USD'] = df_raw['Amount_Currency'].apply(limpiar_monto_a_usd)

rescatados = df_raw['Amount_USD'].notna().sum()
print(f"\u2705 Montos convertidos a USD: {rescatados:,}/{len(df_raw):,}")
print(f"📊 Monto Promedio de Transacción: ${df_raw['Amount_USD'].mean():,.2f} USD")
print(f"📊 Monto Máximo detectado: ${df_raw['Amount_USD'].max():,.2f} USD")
print("\n--- Comparativa original vs limpio ---")
df_raw[['Amount_Currency', 'Amount_USD']].sample(8)


✅ Montos convertidos a USD: 2,441/2,500
📊 Monto Promedio de Transacción: $2,416.37 USD
📊 Monto Máximo detectado: $5,003.43 USD

--- Comparativa original vs limpio ---


,Amount_Currency,Amount_USD
1714,"$ 3,151.68",3151.68
43,"$ 2,939.56",2939.56
2254,¥ 645015,4321.60
2095,"3.050,80 €",3294.86
2400,"$ 2,792.31",2792.31
1230,¥ 698475,4679.78
2341,¥ 643018,4308.22
46,"$ 1,707.37",1707.37


## 🌍 Fase 4: Entity Resolution (Estandarización de Nombres y Países)

Implemento un enfoque de diccionarios Hash en Pandas (`.map`) para consolidar las distintas abreviaturas de un mismo país (US, USA, Estados Unidos) hacia un estándar corporativo (United States).
Adicionalmente, ejecuto limpieza vectorial para remover caracteres extraños en los campos del nombre del comercio (`Merchant`), estandarizando al sector de retail.

In [8]:
mapa_paises = {
    'US': 'United States', 'USA': 'United States', 'Estados Unidos': 'United States',
    'UK': 'United Kingdom', 'GBR': 'United Kingdom', 'Reino Unido': 'United Kingdom',
    'ES': 'Spain', 'España': 'Spain', 'ESP': 'Spain',
    'MX': 'Mexico', 'Mexico': 'Mexico', 'MEX': 'Mexico',
    'JP': 'Japan', 'Japón': 'Japan', 'JPN': 'Japan',
    'PE': 'Peru', 'Perú': 'Peru', 'PER': 'Peru',
    'CO': 'Colombia', 'Colombia': 'Colombia', 'COL': 'Colombia',
}
df_raw['Country'] = df_raw['Country_Raw'].map(mapa_paises).fillna('Unknown')

df_raw['Merchant_clean'] = df_raw['Merchant'].str.lower().str.strip()

df_raw['Merchant_clean'] = df_raw['Merchant_clean'].apply(
    lambda x: re.sub(r'[^a-z0-9 ]', '', str(x)) if pd.notna(x) else x
)

mapa_comercios = {
    'amazon': 'Amazon', 'vtex': 'Vtex', 'mercadolibre': 'MercadoLibre',
    'alibaba': 'Alibaba', 'netflix': 'Netflix', 'uber': 'Uber',
    'airbnb': 'Airbnb', 'shell gas': 'Shell Gas', 'starbucks': 'Starbucks',
    'apple store': 'Apple Store'
}
df_raw['Merchant_clean'] = df_raw['Merchant_clean'].map(mapa_comercios).fillna('Other')

print("=== DISTRIBUCIÓN DE PAÍSES (después de limpiar) ===")
print(df_raw['Country'].value_counts().to_string())
print("\n=== DISTRIBUCIÓN DE COMERCIOS (después de limpiar) ===")
print(df_raw['Merchant_clean'].value_counts().to_string())


=== DISTRIBUCIÓN DE PAÍSES (después de limpiar) ===
Country
United States     474
United Kingdom    431
Peru              336
Spain             293
Japan             284
Mexico            281
Colombia          263
Unknown           138

=== DISTRIBUCIÓN DE COMERCIOS (después de limpiar) ===
Merchant_clean
Airbnb          261
Uber            250
Other           241
Shell Gas       239
Vtex            232
MercadoLibre    223
Netflix         221
Amazon          213
Apple Store     212
Alibaba         211
Starbucks       197


## 🔬 Fase 5: Validación Lógica de Seguridad (Rastreo de IPs)

Los sistemas anti-fraude dependen de ubicaciones de red fiables. He incorporado una función que parsea cada octeto de la dirección IP registrada. Si algún número escapa del rango `0-255`, la bandera `IP_Valid` se marca en Falso (posible suplantación u ocultamiento bot).

Finalmente procedo a purgar (`.dropna`) aquellas transacciones irrecuperables cuyo valor financiero (Monto USD) o fecha no pudieron determinarse tras todos los procesos de limpieza previos.

In [9]:
def validar_ip(ip):
    if pd.isna(ip):
        return False
    partes = str(ip).split('.')
    if len(partes) != 4:
        return False
    try:
        return all(0 <= int(p) <= 255 for p in partes)
    except ValueError:
        return False

df_raw['IP_Valid'] = df_raw['Customer_IP'].apply(validar_ip)
df_raw['Customer_IP_clean'] = df_raw['Customer_IP'].where(df_raw['IP_Valid'], other='IP_INVALIDA')

df_clean = df_raw[[
    'TransactionID',
    'Timestamp_clean',
    'Hour',
    'Merchant_clean',
    'Amount_USD',
    'Country',
    'Customer_IP_clean',
    'IP_Valid'
]].copy()

df_clean.columns = ['transaction_id', 'timestamp', 'hour', 'merchant', 'amount_usd', 'country', 'customer_ip', 'ip_valid']

filas_antes = len(df_clean)
df_clean = df_clean.dropna(subset=['amount_usd', 'timestamp'])
filas_despues = len(df_clean)

print("\u2705 RESUMEN FINAL DEL PROCESO DE LIMPIEZA")
print("=" * 45)
print(f"Filas originales     : {filas_antes:,}")
print(f"Filas eliminadas     : {filas_antes - filas_despues:,} (montos/fechas nulos irrecuperables)")
print(f"Filas limpias finales: {filas_despues:,}")
print(f"IPs inválidas marcadas: {(df_clean['ip_valid'] == False).sum():,}")
print("\n--- Vista previa del DataFrame Limpio ---")
df_clean.sample(5)


✅ RESUMEN FINAL DEL PROCESO DE LIMPIEZA
Filas originales     : 2,500
Filas eliminadas     : 190 (montos/fechas nulos irrecuperables)
Filas limpias finales: 2,310
IPs inválidas marcadas: 354

--- Vista previa del DataFrame Limpio ---


,transaction_id,timestamp,hour,merchant,amount_usd,country,customer_ip,ip_valid
1611,TRX-01612,2023-12-23 05:07:00,5.0,Amazon,1234.43,Peru,35.134.216.32,True
1130,TRX-01131,2023-04-02 09:04:00,9.0,Starbucks,608.99,United Kingdom,IP_INVALIDA,False
1748,TRX-01749,2023-04-16 04:55:00,4.0,Starbucks,3352.14,Colombia,197.68.20.157,True
789,TRX-00790,2023-04-07 03:08:00,3.0,Apple Store,3831.50,Peru,118.27.222.44,True
1441,TRX-01442,2023-11-06 08:55:00,8.0,MercadoLibre,3847.16,Colombia,IP_INVALIDA,False


## 🗄️ Fase 6: LOAD — Construcción de la Base de Datos Relacional (`SQLite`)

El dato está completamente normalizado y validado. Como paso final de la canalización ETL, instancio una base de datos SQL local utilizando la librería `sqlite3` integrada de Python.

Mediante `df.to_sql`, inyecto toda la reportería estructuralmente limpia desde el motor de memoria (Pandas) hacia el motor de disco duro relacional. A partir de aquí, el flujo cambia de Python puro a **Lenguaje SQL** estructurado.

In [10]:
import sqlite3
import os

db_path = '../data/fraude_financiero.db'
conn = sqlite3.connect(db_path)

df_clean.to_sql('transacciones', conn, if_exists='replace', index=False)

cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM transacciones")
total_en_db = cursor.fetchone()[0]

print(f"\u2705 Base de datos creada: {os.path.abspath(db_path)}")
print(f"\u2705 Registros cargados en SQLite: {total_en_db:,}")
print("\n--- Esquema de la tabla SQL ---")
cursor.execute("PRAGMA table_info(transacciones)")
for col in cursor.fetchall():
    print(f"  Columna: {col[1]:<20} Tipo: {col[2]}")


✅ Base de datos creada: d:\Proyectos\proyecto3-etl-fraude\data\fraude_financiero.db
✅ Registros cargados en SQLite: 2,310

--- Esquema de la tabla SQL ---
  Columna: transaction_id       Tipo: TEXT
  Columna: timestamp            Tipo: TIMESTAMP
  Columna: hour                 Tipo: REAL
  Columna: merchant             Tipo: TEXT
  Columna: amount_usd           Tipo: REAL
  Columna: country              Tipo: TEXT
  Columna: customer_ip          Tipo: TEXT
  Columna: ip_valid             Tipo: INTEGER


## 🕵️ Fase 7: ANALYSIS — Detección de Transacciones Ilegítimas (SQL)

Para demostrar viabilidad de negocio mediante integración, las siguientes consultas interactuarán localmente con nuestra Base de Datos `fraude_financiero.db` vía lenguaje puramente analítico (**SQL**).

**Hipótesis de Riesgo N° 1:** Detectar concentración anómala del volumen transaccional en horas de la madrugada (00:00 - 05:00 AM) por bloque geográfico.

In [11]:
query_fraude_nocturno = """
SELECT 
    country                          AS Pais,
    COUNT(*)                         AS Total_Transacciones,
    ROUND(SUM(amount_usd), 2)        AS Volumen_Total_USD,
    ROUND(AVG(amount_usd), 2)        AS Promedio_Por_Transaccion_USD,
    SUM(CASE WHEN ip_valid = 0 THEN 1 ELSE 0 END) AS IPs_Sospechosas
FROM transacciones
WHERE hour BETWEEN 0 AND 5 
  AND amount_usd > 500 
GROUP BY country
ORDER BY Volumen_Total_USD DESC
LIMIT 7;
"""

df_resultado1 = pd.read_sql_query(query_fraude_nocturno, conn)
print("🚨 ALERTA: TOP PAÍSES CON MAYOR VOLUMEN TRANSACCIONAL EN MADRUGADA")
print("="*65)
print(df_resultado1.to_string())


🚨 ALERTA: TOP PAÍSES CON MAYOR VOLUMEN TRANSACCIONAL EN MADRUGADA
             Pais  Total_Transacciones  Volumen_Total_USD  Promedio_Por_Transaccion_USD  IPs_Sospechosas
0   United States                  112          282384.55                       2521.29               19
1  United Kingdom                   87          211745.92                       2433.86               12
2            Peru                   76          200019.83                       2631.84               10
3           Japan                   67          185952.07                       2775.40                8
4           Spain                   64          181649.56                       2838.27                6
5        Colombia                   56          161612.65                       2885.94               12
6          Mexico                   58          145778.68                       2513.43                7


In [12]:
query_bots = """
SELECT 
    merchant                          AS Comercio,
    COUNT(*)                          AS Total_Ops,
    SUM(CASE WHEN ip_valid = 0 THEN 1 ELSE 0 END)  AS Ops_IP_Invalida,
    ROUND(
        100.0 * SUM(CASE WHEN ip_valid = 0 THEN 1 ELSE 0 END) / COUNT(*), 
        1
    )                                 AS Pct_Riesgo
FROM transacciones
GROUP BY merchant
HAVING Ops_IP_Invalida > 10
ORDER BY Pct_Riesgo DESC;
"""

df_resultado2 = pd.read_sql_query(query_bots, conn)
print("🤖 ALERTA: COMERCIOS CON MAYOR % DE OPERACIONES DESDE IPs INVálidas")
print("="*65)
print(df_resultado2.to_string())


🤖 ALERTA: COMERCIOS CON MAYOR % DE OPERACIONES DESDE IPs INVálidas
        Comercio  Total_Ops  Ops_IP_Invalida  Pct_Riesgo
0         Airbnb        239               51        21.3
1        Alibaba        190               33        17.4
2   MercadoLibre        200               33        16.5
3      Shell Gas        220               35        15.9
4        Netflix        207               33        15.9
5           Vtex        217               33        15.2
6    Apple Store        198               29        14.6
7         Amazon        196               28        14.3
8          Other        226               31        13.7
9      Starbucks        184               22        12.0
10          Uber        233               26        11.2


In [13]:
conn.close()
print("\u2705 Análisis completo. Conexión a SQLite cerrada correctamente.")


✅ Análisis completo. Conexión a SQLite cerrada correctamente.
